# Построение матрицы корреляции

**Дисциплина:** Введение в анализ больших данных  
**Задание 6**

Датасет **NMES1988** (медицинское обследование, 4406 наблюдений).

Что делаем:
1. Проверка нормальности количественных переменных (Shapiro–Wilk).
2. Корреляция Спирмена `visits ~ age`.
3. Матрица корреляций + heatmap.
4. Значимость пар с |r| < 0.5 — вся выборка, мужчины, женщины.
5. `visits` vs остальные — с группировкой по региону.

## 0. Импорт и загрузка данных

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
sns.set_style("whitegrid")

In [ ]:
data = pd.read_csv("NMES1988.csv", index_col=0)

cols = ["visits", "health", "chronic", "adl", "region", "age",
        "gender", "married", "school", "income", "employed", "insurance"]
data = data[cols]

print("Размерность:", data.shape)
print("Столбцы:", data.columns.tolist())

quant_cols = ["visits", "chronic", "age", "school", "income"]
print("\nКоличественные переменные:", quant_cols)

---
## 1. Проверка нормальности (Shapiro–Wilk)

H0: выборка из нормального распределения.  
p > 0.05 → H0 не отвергается.  
При большом n тест «ловит» малые отклонения, поэтому дополнительно смотрим гистограммы.

In [ ]:
for col in quant_cols:
    x = data[col].dropna()
    W, p = stats.shapiro(x)
    verdict = "нормально" if p > 0.05 else "НЕ нормально"
    print(f"{col:10s}: W={W:.4f}, p={p:.4e} → {verdict}")

In [ ]:
fig, axes = plt.subplots(1, len(quant_cols), figsize=(3.5 * len(quant_cols), 4))
for ax, col in zip(axes, quant_cols):
    ax.hist(data[col], bins=30, color="steelblue", edgecolor="white")
    ax.set_title(col)
plt.suptitle("Распределения количественных переменных", fontsize=13)
plt.tight_layout()
plt.show()

---
## 2. Корреляция Спирмена: visits ~ age

In [ ]:
rho, p = stats.spearmanr(data["visits"], data["age"], nan_policy="omit")
print(f"rho = {rho:.4f}")
print(f"p   = {p:.4e}")
print(f"Интерпретация: {'значимая' if p < 0.05 else 'незначимая'} связь, "
      f"{'положительная' if rho > 0 else 'отрицательная'}")

---
## 3. Матрица корреляций + heatmap

In [ ]:
corr = data[quant_cols].corr(method="spearman")
corr.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlBu_r", center=0,
            vmin=-1, vmax=1, square=True, ax=axes[0])
axes[0].set_title("Матрица корреляций Спирмена")

sns.heatmap(corr, annot=False, cmap="RdYlBu_r", center=0,
            vmin=-1, vmax=1, square=True, ax=axes[1])
axes[1].set_title("Матрица корреляций (цветами)")
plt.tight_layout()
plt.show()

---
## 4. Значимость коэффициентов |r| < 0.5

Смотрим для всей выборки, отдельно для мужчин и женщин.

In [ ]:
def significant_pairs(df, quant_cols, threshold=0.5):
    rows = []
    for i, c1 in enumerate(quant_cols):
        for c2 in quant_cols[i + 1:]:
            sub = df[[c1, c2]].dropna()
            rho, p = stats.spearmanr(sub[c1], sub[c2])
            if abs(rho) < threshold:
                rows.append({"pair": f"{c1} ~ {c2}", "rho": rho, "p": p,
                             "significant": p < 0.05, "n": len(sub)})
    return pd.DataFrame(rows)

In [ ]:
print("--- Вся выборка ---")
print(significant_pairs(data, quant_cols).round(4).to_string(index=False))

In [ ]:
results_by_gender = {}
for g in data["gender"].dropna().unique():
    sub = data[data["gender"] == g]
    res = significant_pairs(sub, quant_cols)
    results_by_gender[g] = res
    print(f"\n--- gender = {g} (n={len(sub)}) ---")
    print(res.round(4).to_string(index=False))

In [ ]:
if "male" in results_by_gender and "female" in results_by_gender:
    m = results_by_gender["male"].set_index("pair")["rho"]
    f = results_by_gender["female"].set_index("pair")["rho"]
    compare = pd.DataFrame({"male": m, "female": f}).dropna()
    if len(compare) > 0:
        compare.plot(kind="bar", figsize=(10, 5), edgecolor="black")
        plt.title("Сравнение rho для пар |r| < 0.5: мужчины vs женщины")
        plt.ylabel("Spearman rho")
        plt.xticks(rotation=30, ha="right")
        plt.axhline(0, color="black", lw=0.8)
        plt.tight_layout()
        plt.show()

---
## 5. visits vs остальные — по region

In [ ]:
others = ["chronic", "age", "school", "income"]
rows = []
for region in data["region"].dropna().unique():
    sub = data[data["region"] == region]
    for col in others:
        s = sub[["visits", col]].dropna()
        if len(s) < 3:
            continue
        rho, p = stats.spearmanr(s["visits"], s[col])
        rows.append({"region": region, "variable": col, "rho": rho, "p": p, "n": len(s)})

res_region = pd.DataFrame(rows)
res_region.round(4).to_string(index=False)

In [ ]:
pivot = res_region.pivot(index="region", columns="variable", values="rho")
pivot.round(3)

In [ ]:
plt.figure(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdYlBu_r", center=0,
            vmin=-1, vmax=1)
plt.title("Spearman: visits ~ другие переменные, по region")
plt.tight_layout()
plt.show()